# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using SNPE

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Snapdragon Neural Processing Engine (SNPE) SDK**.

Note: This notebook uses the SNPE command-line flow because it is still widely used for DLC-based deployment. For newer projects, Qualcomm’s QAIRT tools provide a more unified interface around conversion, quantization, execution, and profiling.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to SNPE’s Deep Learning Container (DLC) format.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference.
6. **HTP graph compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325).

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries.
import glob
import os
import onnx
import random
import torch
import uuid

import numpy as np

from libreyolo.models.yolox.nn import LibreYOLOXModel
from PIL import Image

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (resized to 640×640, normalized to `[0, 1]`).
- A **representative calibration dataset** in SNPE’s `.raw` binary format. SNPE uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `preprocess()` helper function used throughout this pipeline.

In [2]:
def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    Args:
        original_image (np.ndarray): Input image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image normalized to [0, 1].
    """

    image = Image.fromarray(original_image).convert("RGB")
    image = image.resize((size, size), Image.BILINEAR)

    image = np.asarray(image).astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
    image = np.expand_dims(image, axis=0)   # CHW -> NCHW

    return image

### Preparing the Calibration Dataset

SNPE’s quantization tool (`snpe-dlc-quantize`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(H, W, C)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **1000 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `snpe-dlc-quantize`.

In [3]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 1000

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        image = Image.open(filename)
        original_image = np.array(image)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

SNPE does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which SNPE can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=18`.

The model decodes the grid offsets inside the graph and returns:
[1, 8400, 85]

where:

8400 = 80x80 + 40x40 + 20x20
85 = 4 bbox values + 1 objectness + 80 class probabilities

The bbox format is:

[cx, cy, w, h]

in the resized 640x640 input coordinate space. When use the model, it is necessary the confidence filtering, `cxcywh -> xyxy` conversion, scaling back to the original image, and NMS.

In [4]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../models/snpe", exist_ok=True)

if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

checkpoint = torch.load(
    "../models/LibreYOLOXs.pt",
    map_location="cpu"
)

model = LibreYOLOXModel(config="s", nb_classes=80)
model.load_state_dict(checkpoint["model"], strict=True)

model.eval()
model.head.export = True

dummy = torch.randn(1, 3, 640, 640)
onnx_path = "../models/LibreYOLOXs.onnx"

if not os.path.exists(onnx_path):
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        opset_version=13,
        dynamo=False,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["detections"]
    )

    print(f"Exported decoded ONNX to: {onnx_path}.")

### Validating the ONNX output shape

This cell checks that the exported ONNX has exactly one output named `detections` with shape `[1, 8400, 85]`. If you still see three outputs, the model was exported without `torch_model.head.export = True`.

In [5]:
onnx_model = onnx.load("../models/LibreYOLOXs.onnx")
onnx.checker.check_model(onnx_model)

print("Inputs:")
for tensor in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

assert len(onnx_model.graph.output) == 1, "Expected exactly one decoded output."
assert "detections" in output_shapes, "Expected output named `detections`."
assert output_shapes["detections"] == [1, 8400, 85], f"Unexpected output shape: {output_shapes['detections']}"

print("ONNX export is correct: one decoded output [1, 8400, 85].")

Inputs:
 images: [1, 3, 640, 640]
Outputs:
 detections: [1, 8400, 85]
ONNX export is correct: one decoded output [1, 8400, 85].


## Converting the Model to SNPE DLC Format

SNPE uses its own model format called the **DLC (Deep Learning Container)**. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `snpe-onnx-to-dlc` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [6]:
!bash -c 'snpe-onnx-to-dlc  \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/snpe/LibreYOLOXs_fp32.dlc'

2026-05-10 10:28:54,567 - 278 - INFO - Input shape info 
2026-05-10 10:28:56,164 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-10 10:28:56,165 - 283 - WARNING - Simplified model validation failed
2026-05-10 10:28:56,528 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-10 10:28:56,528 - 283 - WARNING - Simplified model validation failed
2026-05-10 10:28:56,681 - 278 - INFO - INFO_STATIC_RESHAPE: Applying static reshape to /head/Concat_3_output_0: new name /head/Reshape_1_output_0 new shape [1, -1, 2]
2026-05-10 10:28:56,681 - 283 - WARNING - Only numerical type cast is supported. The cast op: /head/Cast will be interpreted at conversion time
2026-05-10 10:28:56,689 - 278 - INFO - INFO_STATIC_RESHAPE: Applying static reshape to /head/Concat_5_output_0: new name /head/Reshape_3_output_0 new shape [1, -1, 2]
2026-05-10 10:28:56,689 - 283 - WARNING - Only numerical type cast is supported. The cast op:

### Inspecting the FP32 DLC

The `snpe-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by SNPE before proceeding to quantization.

In [7]:
!bash -c 'snpe-dlc-info  \
    --input_dlc ../models/snpe/LibreYOLOXs_fp32.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_fp32.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; unroll_gru_time_steps=True; quantization_overrides=; prepare_inputs_as_params=False; perform_axes_to_spatial_first_order=True; unroll_lstm_time_steps=True; packed_max_seq=1; validation_target=[]; packed_masked_softmax_inputs=[]; optimization_pass_mode=ir_optimizer_mainline; op_package_lib=; no_simplification=False; multi_time_steps_gru=False; match_caffe_ssd_to_tf=True; preserve_onnx_output_order=False; perform_layout_transformation=False; keep_quant_nodes=False; input_dtype=[]; keep_disconnected_nodes=False; ir_optimizer_config=; input_encoding=[]; handle_gather_negative_indices=True; copyright_file=None; force_prune_cast_ops=False; use_convert_quantization_nodes=False; package_name=None; defer_loading=False; dry_run=None; input_type=[]; float_bitwidth=32; out_

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `snpe-dlc-quantize` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing an INT8 DLC.

In [8]:
!bash -c 'snpe-dlc-quantize  \
    --input_dlc ../models/snpe/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/snpe/LibreYOLOXs_int8.dlc'

/qairt/2.41.0.251128/bin/x86_64-linux-clang/snpe-dlc-quantize: line 224: [: too many arguments
[INFO] InitializeStderr: DebugLog initialized.
[INFO] Processed command-line arguments
     2.5ms [  INFO ] Inferences will run in sync mode
Processing inference input(s):
raw/7f3d6c81-4ee7-4e5a-9db5-0a4fbde7f5a3.raw
[INFO] Quantized parameters
raw/df20eefc-43d2-4a33-8195-336f13c7a94f.raw
raw/c5facc87-f986-4144-8ddd-056e65663254.raw
raw/8928b015-c7da-4ead-9fa7-f2b3524d6562.raw
raw/b878b7bc-783c-42c6-93a5-af96a6f13229.raw
raw/4e4caa8e-933e-46a3-abb5-c7e652b29284.raw
raw/554ab847-3533-44c6-9efa-d764f1b8bc91.raw
raw/086e39ed-9abb-4b7e-9623-0c11002a1257.raw
raw/4bca67a3-f517-4534-a300-69a2a68c2c94.raw
raw/2ca4ba51-5a2c-4911-89e8-36ebb00a9729.raw
raw/327570f1-ab49-4672-9aa7-05130821120e.raw
raw/a137e2f2-54a5-4902-9864-854dabc730a9.raw
raw/779ae593-f5d8-4bd4-8d93-ea0260917fd1.raw
raw/3540cb8d-1f0c-44d9-a456-d61f44572d74.raw
raw/a844bf4e-0e09-44ae-a2ad-225c44557a52.raw
raw/1dd1773e-60f5-44f6-8bf8-54

### Inspecting the INT8 DLC

After quantization, `snpe-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [9]:
!bash -c 'snpe-dlc-info  \
    --input_dlc ../models/snpe/LibreYOLOXs_int8.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_int8.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; dump_inferred_model=False; dump_ir_optimizer_config_template=False; input_dim=None; converter_op_package_lib=; adjust_nms_features_dims=True; define_symbol=None; squash_box_decoder=True; inject_cast_for_gather=True; extract_color_transform=True; align_matmul_ranks=True; dumpIR=False; dump_custom_io_config_template=; dump_value_info=False; optimization_pass_mode_config=; preprocess_roi_pool_inputs=True; custom_op_config_paths=None; batch=None; use_quantize_v2=False; keep_int64_inputs=False; float_bias_bw=0; disable_batchnorm_folding=False; enable_strict_validation=False; disable_defer_loading=False; enable_framework_trace=False; dump_qairt_io_config_yaml=; preserve_io=[]; debug=-1; multi_time_steps_lstm=False; expand_lstm_op_structure=False; dump_pass_trace_info=

### Compiling the Graph for the Hexagon Tensor Processor (HTP)

Qualcomm® Snapdragon SoCs include a dedicated AI accelerator called the **Hexagon Tensor Processor (HTP)**. To fully leverage HTP hardware, SNPE can perform an **offline graph compilation** step that optimizes the DLC graph layout for a specific target SoC.

The `snpe-dlc-graph-prepare` command takes the INT8 DLC and the target SoC identifier — `sm7325`, corresponding to the **Snapdragon 778G** — and produces a pre-compiled DLC ready for on-device deployment with maximum HTP utilization.

In [10]:
!bash -c 'snpe-dlc-graph-prepare  \
    --input_dlc ../models/snpe/LibreYOLOXs_int8.dlc  \
    --output_dlc ../models/snpe/LibreYOLOXs_int8_sm7325.dlc \
    --htp_socs sm7325'

[INFO] InitializeStderr: DebugLog initialized.
[INFO] SNPE HTP Offline Prepare: Attempting to create cache for SM7325
[USER_INFO] Target device backend record identifier: HTP_V68_SM7325_2MB
[USER_INFO] No cache record in the DLC matches the target device (HTP_V68_SM7325_2MB). Creating a new record
[USER_INFO] Checking unsigned PD session
[INFO] Attempting to open dynamically linked lib: libHtpPrepare.so
[INFO] dlopen libHtpPrepare.so SUCCESS handle 0x564260c308a0
[INFO] Found Interface Provider (v2.31)
[USER_WARNING]  <W> Initializing HtpProvider
[USER_WARNING]  <W> HTP arch will be deprecated, please set SoC id instead.
[USER_WARNING]  <W> Performance Estimates unsupported on soc SM7325
[USER_WARNING]  <W> Cost Based unsupported on soc SM7325
[USER_INFO] Platform option not set
[USER_INFO] Created ctx=0x1 for Graph Id=0 backend=HTP SNPE Id=0x564260a547f8
[USER_INFO] Context [0x1] Setting priority to: default
[USER_INFO] Offline Prepare VTCM size(MB) selected = 0
[USER_INFO] Optimizati

### Inspecting the HTP-Optimized DLC

A final call to `snpe-dlc-info` confirms that the DLC has been compiled with HTP-specific optimizations for the **sm7325** SoC and is ready for on-device deployment.

In [11]:
!bash -c 'snpe-dlc-info \
    --input_dlc ../models/snpe/LibreYOLOXs_int8_sm7325.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_int8_sm7325.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; dump_inferred_model=False; dump_ir_optimizer_config_template=False; input_dim=None; converter_op_package_lib=; adjust_nms_features_dims=True; define_symbol=None; squash_box_decoder=True; inject_cast_for_gather=True; extract_color_transform=True; align_matmul_ranks=True; dumpIR=False; dump_custom_io_config_template=; dump_value_info=False; optimization_pass_mode_config=; preprocess_roi_pool_inputs=True; custom_op_config_paths=None; batch=None; use_quantize_v2=False; keep_int64_inputs=False; float_bias_bw=0; disable_batchnorm_folding=False; enable_strict_validation=False; disable_defer_loading=False; enable_framework_trace=False; dump_qairt_io_config_yaml=; preserve_io=[]; debug=-1; multi_time_steps_lstm=False; expand_lstm_op_structure=False; dump_pass_trac

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.